In [0]:
import pandas as pd
import io
import base64

# Define your catalog and schema here
SCHEMA_NAME = "project.gold"

# Fetch all tables from the specified schema
tables_df = spark.sql(f"SHOW TABLES IN {SCHEMA_NAME}")
table_names = [row.tableName for row in tables_df.collect()]

print("✨ Generating in-memory Parquet download links... Please wait!\n")

for table_name in table_names:
    full_table_path = f"{SCHEMA_NAME}.{table_name}"
    
    # 1. Read the Spark DataFrame
    df_spark = spark.table(full_table_path)
    
    # 2. Convert to Pandas safely using collect() to avoid cluster security restrictions
    data = [row.asDict() for row in df_spark.collect()]
    df_pandas = pd.DataFrame(data)
    
    # 3. Write the Parquet file to an in-memory buffer (io.BytesIO) instead of the restricted disk
    buffer = io.BytesIO()
    df_pandas.to_parquet(buffer, index=False)
    buffer.seek(0)
    
    # 4. Encode the buffer to Base64 so it can be passed directly through the browser
    b64 = base64.b64encode(buffer.read()).decode()
    
    # 5. Generate an HTML download button for the file
    html = f"""
    <a href="data:application/octet-stream;base64,{b64}" 
       download="{table_name}.parquet" 
       style="font-size: 14px; font-weight: bold; color: white; background-color: #0078D4; text-decoration: none; padding: 10px 15px; border-radius: 5px; display: inline-block; margin: 5px 0; font-family: sans-serif;">
       📥 Download {table_name}.parquet
    </a>
    <br>
    """
    displayHTML(html)

print("✅ All done! Click the blue buttons below to download your Parquet files.")